In [ ]:
# =============================================================================
# FACT CRESCENDO ENGLISH: ARTICLE SCRAPER (2022–2025)
# =============================================================================

# -----------------------------
# 1. INSTALL REQUIRED PACKAGES
# -----------------------------
!pip -q install requests beautifulsoup4 pandas tqdm python-dateutil



In [ ]:
# -----------------------------
# 2. IMPORT LIBRARIES
# -----------------------------
import os
import re
import time
import random
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser
from tqdm.auto import tqdm
from google.colab import drive



In [ ]:
# -----------------------------
# 3. MOUNT GOOGLE DRIVE
# -----------------------------
drive.mount("/content/drive")

# Change this folder if you want the project stored elsewhere in Drive.
DRIVE_FOLDER = Path("/content/drive/MyDrive/FactCrescendo_Scraper")
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

# Persistent files: checkpoints survive Colab runtime disconnections.
ARCHIVE_CSV = DRIVE_FOLDER / "factcrescendo_archive_links.csv"
PROGRESS_CSV = DRIVE_FOLDER / "factcrescendo_progress_all_articles.csv"
FINAL_CSV = DRIVE_FOLDER / "factcrescendo_articles_2022_2025.csv"



In [ ]:
# -----------------------------
# 4. SCRAPER SETTINGS
# -----------------------------
BASE_URL = "https://english.factcrescendo.com"

# These are the archive pages to crawl.
START_PAGE = 3
END_PAGE = 16

# Keep only articles with a publication date in this range.
START_YEAR = 2022
END_YEAR = 2025

# Save progress after this many successfully attempted article pages.
CHECKPOINT_EVERY = 10

# Polite delays between requests (seconds).
ARCHIVE_DELAY = (0.7, 1.2)
ARTICLE_DELAY = (0.5, 1.0)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(HEADERS)



In [ ]:
# -----------------------------
# 5. GENERAL HELPER FUNCTIONS
# -----------------------------
def get_soup(url, retries=3, timeout=30):
    """Download a page and return BeautifulSoup; retry transient failures."""
    for attempt in range(1, retries + 1):
        try:
            response = session.get(url, timeout=timeout)
            response.raise_for_status()
            return BeautifulSoup(response.text, "html.parser")
        except requests.RequestException as exc:
            print(f"Request attempt {attempt}/{retries} failed: {url} | {exc}")
            if attempt < retries:
                time.sleep(2 * attempt)
    return None


def clean_text(text):
    """Normalize whitespace and return None for empty text."""
    if text is None:
        return None
    text = re.sub(r"\xa0", " ", str(text))
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    text = text.strip()
    return text or None


def parse_date(date_text):
    """Parse a date string and return YYYY-MM-DD, or None if unparseable."""
    if not date_text:
        return None
    try:
        return parser.parse(str(date_text), fuzzy=True).date().isoformat()
    except (ValueError, TypeError, OverflowError):
        return None


def get_year(date_value):
    """Extract the year from an ISO date string."""
    if not date_value:
        return None
    try:
        return int(str(date_value)[:4])
    except (ValueError, TypeError):
        return None


def save_csv_safely(dataframe, path):
    """Write CSV to a temporary file first, then replace the destination."""
    temporary_path = Path(str(path) + ".tmp")
    dataframe.to_csv(temporary_path, index=False, encoding="utf-8-sig")
    temporary_path.replace(path)




In [ ]:
# -----------------------------
# 6. EXTRACT ARTICLE LINKS FROM AN ARCHIVE PAGE
# -----------------------------
def extract_archive_articles(page_number):
    """Extract article URL, title, archive date, and author from one page."""
    archive_url = f"{BASE_URL}/archive/page/{page_number}/"
    soup = get_soup(archive_url)

    if soup is None:
        return []

    cards = soup.select("article")
    if not cards:
        cards = soup.select(
            ".td_module_wrap, .post, .entry, .archive-item, .jeg_post"
        )

    articles = []
    seen_urls_on_page = set()

    excluded_paths = (
        "/archive/", "/category/", "/author/", "/tag/", "/page/"
    )
    allowed_hosts = {
        "english.factcrescendo.com",
        "www.english.factcrescendo.com",
    }

    for card in cards:
        link = card.select_one(
            "h1 a, h2 a, h3 a, "
            ".entry-title a, .td-module-title a, .jeg_post_title a"
        )
        if not link or not link.get("href"):
            continue

        article_url = urljoin(BASE_URL, link["href"]).split("#")[0]
        parsed_url = urlparse(article_url)

        if parsed_url.netloc.lower() not in allowed_hosts:
            continue
        if any(path in parsed_url.path.lower() for path in excluded_paths):
            continue
        if article_url in seen_urls_on_page:
            continue

        seen_urls_on_page.add(article_url)
        title = clean_text(link.get_text(" ", strip=True))

        # Try to obtain the publication date from the archive card.
        archive_date = None
        time_tag = card.find("time")
        if time_tag:
            archive_date = parse_date(
                time_tag.get("datetime") or time_tag.get_text(" ", strip=True)
            )

        if not archive_date:
            for selector in (
                ".entry-date", ".posted-on", ".post-date",
                ".td-post-date", ".date", "[class*='date']"
            ):
                element = card.select_one(selector)
                if element:
                    archive_date = parse_date(
                        element.get("datetime")
                        or element.get_text(" ", strip=True)
                    )
                    if archive_date:
                        break

        # Try to obtain the author from the archive card.
        archive_author = None
        for selector in (
            ".author a", ".td-post-author-name a", ".post-author a",
            ".entry-author a", "[rel='author']"
        ):
            element = card.select_one(selector)
            if element:
                archive_author = clean_text(element.get_text(" ", strip=True))
                if archive_author:
                    break

        articles.append({
            "archive_page": page_number,
            "archive_title": title,
            "archive_date": archive_date,
            "archive_author": archive_author,
            "url": article_url,
        })

    return articles




In [ ]:
# -----------------------------
# 7. SCRAPE A SINGLE ARTICLE PAGE
# -----------------------------
def scrape_article(article_url):
    """Extract title, author, publication date, body text, and scrape status."""
    soup = get_soup(article_url)

    if soup is None:
        return {
            "title": None,
            "author": None,
            "date": None,
            "content": None,
            "scrape_status": "request_failed",
        }

    # TITLE
    title = None
    for selector in (
        "h1.entry-title", "h1.td-post-title", "h1.post-title",
        ".jeg_post_title", "article h1", "h1"
    ):
        element = soup.select_one(selector)
        if element:
            title = clean_text(element.get_text(" ", strip=True))
            if title:
                break

    if not title:
        og_title = soup.find("meta", property="og:title")
        if og_title:
            title = clean_text(og_title.get("content"))

    # AUTHOR
    author = None
    for selector in (
        ".td-post-author-name a", ".author a", ".author-name",
        ".post-author a", ".entry-author a", ".jeg_meta_author a",
        "[rel='author']"
    ):
        element = soup.select_one(selector)
        if element:
            author = clean_text(element.get_text(" ", strip=True))
            if author:
                break

    if not author:
        author_meta = soup.find("meta", attrs={"name": "author"})
        if author_meta:
            author = clean_text(author_meta.get("content"))

    # PUBLICATION DATE
    date = None
    time_tag = soup.find("time")
    if time_tag:
        date = parse_date(
            time_tag.get("datetime") or time_tag.get_text(" ", strip=True)
        )

    if not date:
        for selector in (
            ".entry-date", ".posted-on", ".td-post-date",
            ".post-date", ".jeg_meta_date", "[class*='date']"
        ):
            element = soup.select_one(selector)
            if element:
                date = parse_date(
                    element.get("datetime")
                    or element.get_text(" ", strip=True)
                )
                if date:
                    break

    # Metadata fallbacks for publication date.
    if not date:
        for attribute, value in (
            ("property", "article:published_time"),
            ("name", "date"),
            ("name", "publish_date"),
            ("name", "pubdate"),
        ):
            element = soup.find("meta", attrs={attribute: value})
            if element and element.get("content"):
                date = parse_date(element.get("content"))
                if date:
                    break

    # ARTICLE BODY
    content = None
    content_selectors = (
        ".td-post-content", ".entry-content", ".post-content",
        ".article-content", ".single-post-content", ".jeg_post_content",
        "article"
    )

    unwanted_selectors = (
        "script", "style", "noscript", "iframe", "nav", "header",
        "footer", "aside", ".sharedaddy", ".share-buttons",
        ".related-posts", ".related", ".comments", ".comment-respond",
        ".advertisement", ".ads", ".td-post-sharing",
        ".td_block_related_posts", ".jeg_post_tags", ".jeg_share_button"
    )

    for selector in content_selectors:
        element = soup.select_one(selector)
        if not element:
            continue

        # Parse a copy so removing non-article elements does not affect soup.
        body_soup = BeautifulSoup(str(element), "html.parser")
        for unwanted_selector in unwanted_selectors:
            for unwanted in body_soup.select(unwanted_selector):
                unwanted.decompose()

        text = clean_text(body_soup.get_text("\n", strip=True))
        if text and len(text) > 200:
            content = text
            break

    # STATUS
    if not title and not author and not date and not content:
        status = "no_fields_extracted"
    elif not content:
        status = "content_missing"
    elif not title:
        status = "title_missing"
    else:
        status = "success"

    return {
        "title": title,
        "author": author,
        "date": date,
        "content": content,
        "scrape_status": status,
    }




In [ ]:
# -----------------------------
# 8. CRAWL ARCHIVES AND SAVE LINK CHECKPOINT
# -----------------------------
if ARCHIVE_CSV.exists():
    archive_df = pd.read_csv(ARCHIVE_CSV).fillna("")
    archive_articles = archive_df.to_dict("records")
    print(f"Loaded {len(archive_articles)} archived article links from Drive.")
else:
    archive_articles = []
    seen_urls = set()

    for page_number in tqdm(
        range(START_PAGE, END_PAGE + 1), desc="Crawling archive pages"
    ):
        page_articles = extract_archive_articles(page_number)

        for article in page_articles:
            if article["url"] not in seen_urls:
                seen_urls.add(article["url"])
                archive_articles.append(article)

        print(f"Archive page {page_number}: {len(page_articles)} links found.")
        time.sleep(random.uniform(*ARCHIVE_DELAY))

    archive_df = pd.DataFrame(
        archive_articles,
        columns=[
            "archive_page", "archive_title", "archive_date",
            "archive_author", "url"
        ],
    )
    save_csv_safely(archive_df, ARCHIVE_CSV)
    print(f"Saved archive link checkpoint: {ARCHIVE_CSV}")

# Normalize URLs and remove duplicates in the archive list.
archive_df = pd.DataFrame(archive_articles)
if not archive_df.empty and "url" in archive_df.columns:
    archive_df = archive_df.drop_duplicates(subset="url", keep="first")
    archive_articles = archive_df.to_dict("records")

print("Total unique archive article URLs:", len(archive_articles))




In [ ]:
# -----------------------------
# 9. LOAD EXISTING ARTICLE CHECKPOINT (RESUME)
# -----------------------------
result_columns = [
    "title", "author", "date", "content", "url",
    "archive_page", "scrape_status"
]

if PROGRESS_CSV.exists():
    progress_df = pd.read_csv(PROGRESS_CSV).fillna("")
    # Ensure expected columns exist if an older checkpoint is being reused.
    for column in result_columns:
        if column not in progress_df.columns:
            progress_df[column] = ""
    progress_df = progress_df[result_columns]
    print(f"Loaded {len(progress_df)} processed records from checkpoint.")
else:
    progress_df = pd.DataFrame(columns=result_columns)
    print("No article checkpoint found; starting a new scrape.")

processed_urls = set(progress_df["url"].astype(str))
remaining_articles = [
    article for article in archive_articles
    if str(article.get("url", "")) not in processed_urls
]

print("Already processed:", len(processed_urls))
print("Remaining to scrape:", len(remaining_articles))




In [ ]:
# -----------------------------
# 10. SCRAPE REMAINING ARTICLES + PERIODIC CHECKPOINTS
# -----------------------------
new_results = []

for index, article in enumerate(
    tqdm(remaining_articles, desc="Scraping article pages"), start=1
):
    article_url = str(article.get("url", "")).strip()
    if not article_url:
        continue

    result = scrape_article(article_url)

    # Prefer the article-page date; use the archive date if unavailable.
    final_date = result["date"] or article.get("archive_date") or None
    year = get_year(final_date)

    # Record each attempted URL, including undated/out-of-period articles.
    # This prevents repeatedly retrying them when resuming the same run.
    new_results.append({
        "title": result["title"] or article.get("archive_title") or None,
        "author": result["author"] or article.get("archive_author") or None,
        "date": final_date,
        "content": result["content"],
        "url": article_url,
        "archive_page": article.get("archive_page"),
        "scrape_status": result["scrape_status"],
    })

    # Periodically merge and save all attempted article records to Drive.
    if len(new_results) >= CHECKPOINT_EVERY:
        progress_df = pd.concat(
            [progress_df, pd.DataFrame(new_results, columns=result_columns)],
            ignore_index=True,
        )
        progress_df = progress_df.drop_duplicates(subset="url", keep="last")
        save_csv_safely(progress_df, PROGRESS_CSV)
        new_results = []
        print(f"Checkpoint saved: {len(progress_df)} article attempts.")

    time.sleep(random.uniform(*ARTICLE_DELAY))

# Save any remaining records after the loop.
if new_results:
    progress_df = pd.concat(
        [progress_df, pd.DataFrame(new_results, columns=result_columns)],
        ignore_index=True,
    )
    progress_df = progress_df.drop_duplicates(subset="url", keep="last")

save_csv_safely(progress_df, PROGRESS_CSV)
print(f"Full progress checkpoint saved to: {PROGRESS_CSV}")




In [ ]:
# -----------------------------
# 11. FILTER FINAL DATASET TO 2022–2025
# -----------------------------
progress_df["year"] = progress_df["date"].apply(get_year)

df_final = progress_df[
    progress_df["year"].notna()
    & progress_df["year"].between(START_YEAR, END_YEAR)
].copy()

df_final = df_final[result_columns].drop_duplicates(subset="url", keep="last")
df_final = df_final.sort_values(["date", "url"], na_position="last")

save_csv_safely(df_final, FINAL_CSV)

print("-" * 60)
print(f"Articles dated {START_YEAR}–{END_YEAR}: {len(df_final)}")
print(f"Progress checkpoint: {PROGRESS_CSV}")
print(f"Final filtered CSV: {FINAL_CSV}")
print("-" * 60)

display(df_final.head())

# Files are saved directly in Google Drive. No download prompt is necessary.